In [1]:
import io
import numpy as np
from PIL import Image
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, regexp_extract
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, ArrayType

spark = SparkSession.builder \
    .appName("APTOS_Ingesta_Preprocesamiento") \
    .getOrCreate()

In [2]:
labels_df = spark.read.csv("../data/raw/train.csv", header=True, inferSchema=True)

In [3]:
images_df = spark.read.format("binaryFile").load("../data/raw/train_images/*.png")

images_df = images_df.withColumn("id_code", regexp_extract(col("path"), r"([^/]+)\.png$", 1))

In [4]:
schema = StructType([
    StructField("width", IntegerType(), False),
    StructField("height", IntegerType(), False),
    StructField("mean_r", FloatType(), False),
    StructField("mean_g", FloatType(), False),
    StructField("mean_b", FloatType(), False),
    StructField("image_data", ArrayType(FloatType()), False)
])

def process_image(content):
    try:
        img = Image.open(io.BytesIO(content)).convert('RGB')
        w, h = img.size
        img_resized = img.resize((224, 224))
        img_array = np.array(img_resized) / 255.0
        mean_r = float(np.mean(img_array[:, :, 0]))
        mean_g = float(np.mean(img_array[:, :, 1]))
        mean_b = float(np.mean(img_array[:, :, 2]))
        img_flat = img_array.flatten().tolist()
        return (w, h, mean_r, mean_g, mean_b, img_flat)
    except Exception:
        return None

process_image_udf = udf(process_image, schema)

In [5]:
processed_df = images_df.withColumn("processed", process_image_udf(col("content")))

final_df = processed_df.select(
    col("id_code"),
    col("processed.width").alias("width"),
    col("processed.height").alias("height"),
    col("processed.mean_r").alias("mean_r"),
    col("processed.mean_g").alias("mean_g"),
    col("processed.mean_b").alias("mean_b"),
    col("processed.image_data").alias("image_data")
).join(labels_df, on="id_code", how="inner")

In [6]:
final_count = final_df.count()
print(f"Total de imágenes procesadas: {final_count}")

final_df.show(5)

Total de imágenes procesadas: 3662
+------------+-----+------+----------+----------+-----------+--------------------+---------+
|     id_code|width|height|    mean_r|    mean_g|     mean_b|          image_data|diagnosis|
+------------+-----+------+----------+----------+-----------+--------------------+---------+
|e265c870f9b3| 3388|  2588| 0.3588165|0.25851208| 0.18687046|[0.003921569, 0.0...|        2|
|d85a842d20bd| 3388|  2588|0.37000895|0.16831413| 0.03474429|[0.003921569, 0.0...|        2|
|46f56c38051f| 3388|  2588|0.48240757| 0.2498787| 0.12641908|[0.003921569, 0.0...|        2|
|a150ff5dfe07| 3388|  2588|0.40939862|0.21113656|0.048099473|[0.003921569, 0.0...|        2|
|1fd5d860d4d7| 3388|  2588|0.43962485|0.20737177| 0.05379941|[0.003921569, 0.0...|        4|
+------------+-----+------+----------+----------+-----------+--------------------+---------+
only showing top 5 rows

